In [1]:
import rby1_sdk as rby
from dynamixel_sdk import PortHandler, PacketHandler, COMM_SUCCESS
import time
from typing import Optional, List
import threading
import numpy as np
import functions as fn

In [2]:
# --- Gripper ------------------------------------ # 
PROTOCOL_VERSION = 2.0
BAUDRATE = 2000000
DEVICE_NAME_GRIPPER = "/dev/rby1_gripper"

ADDR_OPERATING_MODE     = 11                # 0: Current Control / 1: Velocity Control / 3: Position Control(default)
ADDR_TORQUE_ENABLE      = 64                # 0: Turn OFF(default) / 1: Turn ON
ADDR_LED                = 65                # 0: Turn OFF(default) / 1: Turn ON
ADDR_GOAL_CURRENT       = 102
ADDR_GOAL_POSITION      = 116   
ADDR_PRESENT_CURRENT    = 126               # Unit: 2.69mA
ADDR_PRESENT_POSITION   = 132               # Unit: 1-pulse(or Tick)

CURRENT_CONTROL_MODE    = 0
POSTION_CONTROL_MODE    = 3
TORQUE_DISABLE          = 0
TORQUE_ENABLE           = 1

DXL_MIN_POS_VALUE       = 0
DXL_MAX_POS_VALUE       = 4095

RESOLUTION_TICKS_PER_REV    = 4096.0
TWO_PI                      = 2.0 * np.pi
RAD2DEG                     = 180.0 / np.pi

R_DXL_ID                    = 0
L_DXL_ID                    = 1

is_init_finish              = False

In [3]:
ROBOT_ADDRESS = "localhost:50051"           # For Docker Simulation
# ROBOT_ADDRESS = "192.168.30.1:50051"        # For Real Robot

robot = rby.create_robot_a(ROBOT_ADDRESS)   # Robot Model is "A"
model = robot.model()
if not robot.connect():
    print("Could not connect to robot")
    exit(1)
else:
    print("SUCCESS: Connect to the robot.")
    if (robot.power_on(".*") == True):         # Turn on all the power (5v, 12v, 24v, 48v)
        time.sleep(1)
        if (robot.servo_on(".*") == True):         # Enable servo motors on all actuators
            time.sleep(1)
            robot.reset_fault_control_manager()
            robot.enable_control_manager(unlimited_mode_enabled=True)
            print("SUCCESS: Power ON the Robot.")
            
            # 여기 추가: 그리퍼 쪽 12V 출력 ON
            time.sleep(1)
            robot.set_tool_flange_output_voltage("right", 12)
            robot.set_tool_flange_output_voltage("left", 12)
            print("SUCCESS: Power ON the Gripper.")
        else:
            print("Failed to power ON the Servo")

SUCCESS: Connect to the robot.
Failed to power ON the Servo


In [4]:
# port_hdlr = PortHandler(DEVICE_NAME_GRIPPER)
# packet_hdlr = PacketHandler(PROTOCOL_VERSION)

# if port_hdlr.openPort():
#     print(f"SUCCESS: open the port: {DEVICE_NAME_GRIPPER}")
# else:
#     raise RuntimeError(f"Failed to open the port: {DEVICE_NAME_GRIPPER}")

# if port_hdlr.setBaudRate(BAUDRATE):
#     print(f"SUCCESS: change the baudrate {BAUDRATE}")
# else:
#     raise RuntimeError(f"Failed to change the baudrate")

In [5]:
Desired_Pos = fn.PACK_POS      # ZERO_POS, READY_POS, PACK_POS

fn.move_position(robot, Desired_Pos, minimum_time=10)

True

In [6]:
Desired_Pos = fn.READY_POS      # ZERO_POS, READY_POS, PACK_POS

fn.move_position(robot, Desired_Pos, minimum_time=10)

True

In [8]:
right_deltas = [
    [0, -0.2, 0],
    [0, 0.2, 0],
    [0, 0, 0.4],
    [0, 0, -0.4]
]

stream = robot.create_command_stream()
for delta in right_deltas:
    stream.send_command(fn.move_endeffector(robot, fn.RIGHT_END, delta))
    time.sleep(2.0)
stream.cancel()
stream.wait()